# Fábrica de vídeos con VoxCPM2 en Google Colab (0 € si hay GPU gratuita)

**Opción de voz preferida, no 100 % autónoma.** Se ejecuta en el navegador al pulsar **Entorno de ejecución → Ejecutar todo**. Google puede no ofrecer GPU gratuita o desconectar la sesión.

La opción de GitHub Actions del ZIP sí puede programarse, pero usa **Gemini TTS** porque GitHub no da GPU en runners gratuitos estándar.

La locución conserva la descripción de anciano ronco de tu prueba, pero no imita la voz de una persona real.


In [ ]:
# 1. Conectar una GPU en Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 (si aparece).
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core >/dev/null
%pip -q install voxcpm soundfile requests beautifulsoup4 google-api-python-client google-auth google-auth-httplib2
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
else: print('Sin GPU: VoxCPM2 puede tardar muchísimo o fallar. No es recomendable continuar.')


In [ ]:
# 2. Sube el ZIP del proyecto descargado desde ChatGPT. No subas tus claves.
from google.colab import files
import zipfile, pathlib, os
uploaded=files.upload()
name=next((n for n in uploaded if n.endswith('.zip')), None)
assert name, 'Falta el ZIP de la Fábrica de Vídeos'
folder=pathlib.Path('/content/FabricaVideoGratis');folder.mkdir(exist_ok=True)
with zipfile.ZipFile(name) as f: f.extractall(folder)
assert (folder/'app/pipeline.py').exists(), 'ZIP incorrecto: debe contener app/pipeline.py'
print('Proyecto listo:',folder)


In [ ]:
# 3. Introduce la clave de Gemini (NO se muestran en pantalla).
from getpass import getpass
os.environ['GEMINI_API_KEY']=getpass('Gemini API Key gratuita: ')
os.environ['VOICE_PROVIDER']='voxcpm'
os.environ['YOUTUBE_UPLOAD']='false'
os.environ['VOX_VOICE_DESCRIPTION']='(An elderly man, rough scratchy voice, worn out and hoarse, grumpy but funny, unhurried)'


In [ ]:
# 4. Generar guion, narración VoxCPM2, imágenes y vídeos reales Wikimedia Commons y montar el MP4.
import subprocess, sys, glob
p=subprocess.run([sys.executable,'-m','app.pipeline'],cwd=str(folder),env=os.environ.copy())
print('Resultado:', 'OK' if p.returncode==0 else 'Falló; consulta el registro anterior')


In [ ]:
# 5. Descargar el MP4 si el render terminó correctamente.
from google.colab import files
from pathlib import Path
out=sorted(folder.glob('output/*/video_final.mp4'))
if out: files.download(str(out[-1]))
else: print('Todavía no hay MP4 final. Revisa el error anterior.')
